# Multi-device VQC comparison

This notebook **joins the per-device summary CSVs** written by the single-device
analysis notebooks (same columns as `vqc_ibm_kingston_summary.csv` / Braket IQMs /
Rigetti, where available). It aggregates the **same hardware-vs-ideal metrics**
already used per device—parity error, entropy error, XEB-style purity gap,
readout spread, classification agreement—and adds **cross-device views** (bars,
heatmaps, overlaid curves, scatter vs ideal).

The registry below also includes the noiseless **Amazon Braket SV1 and TN1
simulators**; they serve as a *sampling-noise reference floor* for the
hardware backends (the residual error magnitudes you see on SV1/TN1 are purely
finite-shot sampling, not device noise).

## How to add another device later

1. Produce a summary CSV with the **same logical columns** (at minimum: `idx` or
   `test_index`, `<Z^10>`, `ideal_Z_all_expectation`, `Z_err_empirical_minus_ideal`,
   `H[bits]_bits`, `ideal_entropy_bits`, `entropy_err_vs_ideal`,
   `xeb_linear_2n_Ep2`, `ideal_linear_xeb_purity`, `purity_delta_hw_minus_ideal`,
   `readout_spread`, `unique_bitstrings`, `correct_vs_true_label`,
   `correct_vs_ideal_prediction`, `true_label`, `has_ideal_ref`). Extra columns are fine.
2. Drop the CSV in `data/processed/`.
3. Append one dict to **`DEVICE_REGISTRY`** in the setup cell:
   `device_key` (short id), `display_name` (plot legend), `summary_csv` (filename),
   optional `family` (e.g. `"IBM"` / `"Braket"` for colour grouping).
4. **Restart kernel & run all.**

The combined long table is also exported to `vqc_multi_device_summary_long.csv` for
paper plots or external stats.

**Section 9** loads dedicated **linear XEB** summaries (`summary.json` +
optional `per_depth_per_circuit.json`) for Braket and IonQ native runs on the
shared `xeb_d01_c*.qasm` circuits.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-darkgrid")

# -----------------------------------------------------------------------------
# Extend this list when new device summaries exist (same folder as this notebook).
# -----------------------------------------------------------------------------
DEVICE_REGISTRY = [
    {"device_key": "ibm_kingston", "display_name": "IBM Kingston", "summary_csv": "vqc_ibm_kingston_summary.csv", "family": "IBM"},
    {"device_key": "ibm_fez", "display_name": "IBM Fez", "summary_csv": "vqc_ibm_fez_summary.csv", "family": "IBM"},
    {"device_key": "ibm_marrakesh", "display_name": "IBM Marrakesh", "summary_csv": "vqc_ibm_marrakesh_summary.csv", "family": "IBM"},
    {"device_key": "iqm_emerald", "display_name": "IQM Emerald (Braket)", "summary_csv": "vqc_braket_iqm_emerald_summary.csv", "family": "Braket"},
    {"device_key": "iqm_garnet", "display_name": "IQM Garnet (Braket)", "summary_csv": "vqc_braket_iqm_garnet_summary.csv", "family": "Braket"},
    {"device_key": "rigetti_cepheus", "display_name": "Rigetti Cepheus (Braket)", "summary_csv": "vqc_braket_rigetti_cepheus_summary.csv", "family": "Braket"},
    {"device_key": "braket_sv1", "display_name": "Braket SV1 (Amazon sim)", "summary_csv": "vqc_braket_sv1_summary.csv", "family": "Braket sim"},
    {"device_key": "braket_tn1", "display_name": "Braket TN1 (Amazon sim)", "summary_csv": "vqc_braket_tn1_summary.csv", "family": "Braket sim"},
    {"device_key": "ionq_native_simulator", "display_name": "IonQ simulator (native cloud)", "summary_csv": "vqc_ionq_native_simulator_summary.csv", "family": "IonQ native"},
    {"device_key": "ionq_native_forte_1", "display_name": "IonQ Forte-1 (native cloud)", "summary_csv": "vqc_ionq_native_forte_1_summary.csv", "family": "IonQ native"},
]

# Canonical circuit index for fair comparison (0–10 QVC tests).
CIRCUIT_IDX_MIN = 0
CIRCUIT_IDX_MAX = 10

# Columns used repeatedly (must exist in merged frame after load; missing -> NaN).
COL_Z = "<Z^10>"
COL_Z_IDEAL = "ideal_Z_all_expectation"
COL_Z_ERR = "Z_err_empirical_minus_ideal"
COL_H = "H[bits]_bits"
COL_H_IDEAL = "ideal_entropy_bits"
COL_ENT_ERR = "entropy_err_vs_ideal"
COL_PUR_HW = "xeb_linear_2n_Ep2"
COL_PUR_ID = "ideal_linear_xeb_purity"
COL_PUR_DELTA = "purity_delta_hw_minus_ideal"
COL_RO = "readout_spread"
COL_UNIQ = "unique_bitstrings"
COL_HW = "mean_HW"
COL_MARGIN = "|<Z^10>|"
COL_CORR_LABEL = "correct_vs_true_label"
COL_CORR_IDEAL = "correct_vs_ideal_prediction"


def resolve_data_dir() -> Path:
    # Processed summary CSVs live under data/processed/.
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve() / "data" / "processed",
        Path.cwd().resolve().parent / "data" / "processed",
        Path.cwd().resolve() / "analysis",
        Path.cwd().resolve().parent / "analysis",
    ]
    for c in candidates:
        if (c / DEVICE_REGISTRY[0]["summary_csv"]).is_file():
            return c
    raise FileNotFoundError(
        "Could not find summary CSVs under data/processed/. Run from repo root or analysis/."
    )


def resolve_repo_root() -> Path:
    for c in [Path.cwd().resolve(), Path.cwd().resolve().parent]:
        if (c / "data" / "processed").is_dir() and (c / "experiments").is_dir():
            return c
    data_dir = resolve_data_dir()
    if data_dir.name == "processed":
        return data_dir.parent.parent
    return data_dir.parent


DATA_DIR = resolve_data_dir()
REPO_ROOT = resolve_repo_root()
print("DATA_DIR =", DATA_DIR)
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
def circuit_index(row: pd.Series) -> int:
    # Canonical circuit index: prefer test_index when present, else folder idx.
    if "test_index" in row.index and pd.notna(row["test_index"]):
        return int(row["test_index"])
    return int(row["idx"])


def load_one(entry: dict, base: Path) -> pd.DataFrame:
    path = base / entry["summary_csv"]
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df.insert(0, "family", entry.get("family", "other"))
    df.insert(0, "display_name", entry["display_name"])
    df.insert(0, "device_key", entry["device_key"])
    df["circuit_idx"] = df.apply(circuit_index, axis=1)
    mask = df["circuit_idx"].between(CIRCUIT_IDX_MIN, CIRCUIT_IDX_MAX)
    df = df.loc[mask].copy()
    if "has_ideal_ref" in df.columns:
        df = df[df["has_ideal_ref"].fillna(False)].copy()
    return df


def load_all_devices(base: Path) -> pd.DataFrame:
    frames = []
    for entry in DEVICE_REGISTRY:
        try:
            frames.append(load_one(entry, base))
        except FileNotFoundError as e:
            print("SKIP (missing file):", e)
    if not frames:
        raise RuntimeError("No device summaries loaded — check DEVICE_REGISTRY paths.")
    all_df = pd.concat(frames, ignore_index=True)
    order = [e["device_key"] for e in DEVICE_REGISTRY]
    all_df["display_name"] = pd.Categorical(
        all_df["display_name"], categories=[e["display_name"] for e in DEVICE_REGISTRY], ordered=True
    )
    all_df["device_key"] = pd.Categorical(all_df["device_key"], categories=order, ordered=True)
    return all_df.sort_values(["circuit_idx", "device_key"]).reset_index(drop=True)


all_df = load_all_devices(DATA_DIR)
print("rows:", len(all_df), "| devices:", all_df["device_key"].nunique(), "| circuits:", sorted(all_df["circuit_idx"].unique()))
all_df.head(8)


## 1. Per-device aggregates

Mean absolute parity error vs ideal, mean entropy error, mean purity gap (hardware
linear XEB proxy minus ideal), mean readout spread, mean number of unique bitstrings,
and classification rates (vs stripe label and vs ideal parity sign).


In [ ]:
def device_aggregate(g: pd.DataFrame) -> pd.Series:
    g2 = g.copy()
    out = {
        "n": len(g2),
        "mean_abs_Z_err": g2[COL_Z_ERR].abs().mean(),
        "rmse_Z_err": np.sqrt((g2[COL_Z_ERR] ** 2).mean()),
        "mean_abs_ent_err": g2[COL_ENT_ERR].abs().mean(),
        "mean_pur_delta": g2[COL_PUR_DELTA].mean(),
        "mean_readout_spread": g2[COL_RO].mean(),
        "mean_unique_bitstrings": g2[COL_UNIQ].mean(),
        "mean_margin": g2[COL_MARGIN].mean(),
    }
    if COL_CORR_LABEL in g2.columns:
        out["acc_vs_true_label"] = g2[COL_CORR_LABEL].astype(bool).mean()
    else:
        out["acc_vs_true_label"] = np.nan
    if COL_CORR_IDEAL in g2.columns:
        out["acc_vs_ideal_pred"] = g2[COL_CORR_IDEAL].astype(bool).mean()
    else:
        out["acc_vs_ideal_pred"] = np.nan
    return pd.Series(out)


agg = all_df.groupby("display_name", observed=False).apply(device_aggregate).reset_index()
agg = agg.sort_values("mean_abs_Z_err")
agg_round = agg.round(
    {
        "mean_abs_Z_err": 4,
        "rmse_Z_err": 4,
        "mean_abs_ent_err": 4,
        "mean_pur_delta": 3,
        "mean_readout_spread": 4,
        "mean_unique_bitstrings": 1,
        "mean_margin": 4,
        "acc_vs_true_label": 3,
        "acc_vs_ideal_pred": 3,
    }
)
agg_round


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
pal = sns.color_palette("husl", n_colors=len(DEVICE_REGISTRY))

x = np.arange(len(agg))
w = 0.35

axes[0, 0].bar(x, agg["mean_abs_Z_err"], color=pal)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(agg["display_name"], rotation=35, ha="right")
axes[0, 0].set_ylabel("mean |Z_err|")
axes[0, 0].set_title("Parity error magnitude vs ideal (lower is better)")

axes[0, 1].bar(x, agg["mean_abs_ent_err"], color=pal)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(agg["display_name"], rotation=35, ha="right")
axes[0, 1].set_ylabel("mean |entropy err|")
axes[0, 1].set_title("Output entropy vs ideal (lower is better)")

axes[1, 0].bar(x, agg["mean_pur_delta"], color=pal)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(agg["display_name"], rotation=35, ha="right")
axes[1, 0].set_ylabel("mean purity delta (HW − ideal)")
axes[1, 0].set_title("XEB-style linear purity gap (context: sign can vary)")

axes[1, 1].bar(x - w / 2, agg["acc_vs_true_label"], width=w, label="vs true stripe", color="#4c72b0")
axes[1, 1].bar(x + w / 2, agg["acc_vs_ideal_pred"], width=w, label="vs ideal sign", color="#dd8452")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(agg["display_name"], rotation=35, ha="right")
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].set_ylabel("fraction correct")
axes[1, 1].set_title("Classification accuracy (per-job majority)")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 2. Same metric, every circuit (heatmap)

Rows: **circuit index** (`circuit_idx`). Columns: **device**. Colour shows
`Z_err_empirical_minus_ideal` (empirical parity minus ideal). Missing cells stay
empty (e.g. if a device skipped a test).


In [ ]:
pivot_z = all_df.pivot_table(
    index="circuit_idx", columns="device_key", values=COL_Z_ERR, aggfunc="first"
)
plt.figure(figsize=(11, 5.5))
sns.heatmap(pivot_z.T, cmap="RdBu_r", center=0.0, linewidths=0.5, cbar_kws={"label": COL_Z_ERR})
plt.title("Parity error vs ideal by device and circuit")
plt.xlabel("circuit_idx")
plt.ylabel("device_key")
plt.tight_layout()
plt.show()


## 3. Empirical vs ideal parity (scatter)

Each point is one circuit on one device. Perfect agreement lies on the diagonal.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))
for dk, sub in all_df.groupby("device_key", observed=False):
    ax.scatter(
        sub[COL_Z_IDEAL],
        sub[COL_Z],
        s=55,
        alpha=0.85,
        label=str(dk),
    )
lims = -1.05, 1.05
ax.plot(lims, lims, "k--", lw=1, alpha=0.5)
ax.set_xlim(*lims)
ax.set_ylim(*lims)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("ideal ⟨Z^10⟩")
ax.set_ylabel("hardware ⟨Z^10⟩")
ax.set_title("Hardware vs noiseless parity expectation")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 4. Curves across circuits

Hardware ⟨Z^10⟩ vs `circuit_idx` for each device (same x-axis for direct overlay).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, sub in all_df.groupby("display_name", observed=False):
    sub2 = sub.sort_values("circuit_idx")
    ax.plot(sub2["circuit_idx"], sub2[COL_Z], marker="o", ms=4, lw=1.5, label=name)
ideal_curve = (
    all_df.drop_duplicates("circuit_idx")
    .sort_values("circuit_idx")
    .set_index("circuit_idx")[COL_Z_IDEAL]
)
ax.plot(ideal_curve.index, ideal_curve.values, "k--", lw=2, alpha=0.55, label="ideal (OpenQASM)")
ax.axhline(0, color="gray", lw=0.8, alpha=0.5)
ax.set_xlabel("circuit_idx")
ax.set_ylabel("⟨Z^10⟩")
ax.set_title("Parity expectation trajectories")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Distribution of |Z_err| across circuits

Violin/box view summarises spread **within** each device across the QVC tests.


In [ ]:
plot_df = all_df.assign(abs_Z_err=all_df[COL_Z_ERR].abs())
plt.figure(figsize=(10, 4.5))
sns.violinplot(data=plot_df, x="display_name", y="abs_Z_err", inner="box", cut=0)
plt.xticks(rotation=30, ha="right")
plt.ylabel("|Z_err| vs ideal")
plt.xlabel("")
plt.title("Across-circuit spread of parity error magnitude")
plt.tight_layout()
plt.show()


## 6. Readout spread & output diversity

**Readout spread** (max − min marginal P1) and **unique bitstrings** (support size)
summarise how spread out the 1024-shot empirical distribution is.


In [ ]:
sub = all_df.groupby("display_name", observed=False)[[COL_RO, COL_UNIQ]].mean().reset_index()

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(sub["display_name"], sub[COL_RO], color=sns.color_palette("muted", n_colors=len(sub)))
ax.set_xticklabels(sub["display_name"], rotation=30, ha="right")
ax.set_ylabel("mean readout spread")
ax.set_title("Readout spread (device means over circuits)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(sub["display_name"], sub[COL_UNIQ], color=sns.color_palette("pastel", n_colors=len(sub)))
ax.set_xticklabels(sub["display_name"], rotation=30, ha="right")
ax.set_ylabel("mean unique bitstrings")
ax.set_title("Output diversity (device means over circuits)")
plt.tight_layout()
plt.show()


## 7. Cross-device disagreement on the same circuit

For each `circuit_idx`, empirical **std** of hardware ⟨Z^10⟩ across devices (after
mean-centering optional). High values highlight circuits where backends diverge.


In [ ]:
by_c = all_df.groupby("circuit_idx")[COL_Z].agg(["mean", "std", "min", "max"]).reset_index()
by_c["range_Z"] = by_c["max"] - by_c["min"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_c["circuit_idx"], by_c["std"], color="steelblue", alpha=0.85)
ax.set_xlabel("circuit_idx")
ax.set_ylabel("std of ⟨Z^10⟩ across devices")
ax.set_title("Cross-device spread of empirical parity (same circuit)")
plt.tight_layout()
plt.show()

by_c


## 8. Export combined long table

One row per (device, circuit); use this file in other notebooks, R, or plotting tools.


In [ ]:
out_long = DATA_DIR / "vqc_multi_device_summary_long.csv"
all_df.to_csv(out_long, index=False)
print("Wrote", out_long.resolve(), "rows:", len(all_df))


## 9. Linear XEB benchmark (random circuits)

Dedicated **cross-entropy benchmarking** on the shared `xeb_d01_c*.qasm` set (depth tag
`d01` in the filename). This is separate from the VQC purity proxy in sections 1–7:
here we compare **linear XEB fidelity** \(F_{\mathrm{XEB}}\) (mean ± SEM over
circuits at each depth).

Paths are relative to the repo root. Append entries to
**`XEB_DEVICE_REGISTRY`** when new `summary.json` / `per_depth_per_circuit.json`
exports land.


In [ ]:
import json
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
ANALYSIS = ROOT / "analysis"
EXPERIMENTS = ROOT / "experiments"


# summary_json / per_circuit_json paths are relative to REPO_ROOT.
XEB_DEVICE_REGISTRY = [
    {
        "device_key": "braket_emerald",
        "display_name": "IQM Emerald (Braket XEB)",
        "summary_json": "data/raw/braket_emerald/summary.json",
        "per_circuit_json": None,
        "family": "Braket",
    },
    {
        "device_key": "braket_tn1",
        "display_name": "Braket TN1 (sim)",
        "summary_json": "data/raw/braket_tn1/xeb/summary.json",
        "per_circuit_json": "data/raw/braket_tn1/xeb/per_depth_per_circuit.json",
        "family": "Braket sim",
    },
    {
        "device_key": "ionq_native_simulator",
        "display_name": "IonQ simulator (native cloud)",
        "summary_json": "data/raw/ionq_native/20260708_143259Z/simulator_ideal/xeb/summary.json",
        "per_circuit_json": "data/raw/ionq_native/20260708_143259Z/simulator_ideal/xeb/per_depth_per_circuit.json",
        "family": "IonQ native",
    },
    {
        "device_key": "ionq_native_forte_1",
        "display_name": "IonQ Forte-1 (native cloud)",
        "summary_json": "data/raw/ionq_native/20260709_070901Z/qpu_forte_1/xeb/summary.json",
        "per_circuit_json": "data/raw/ionq_native/20260709_070901Z/qpu_forte_1/xeb/per_depth_per_circuit.json",
        "family": "IonQ native",
    },
]


def _is_xeb_summary(d: dict) -> bool:
    b = str(d.get("benchmark", ""))
    return b == "xeb" or b.startswith("xeb_")


def load_xeb_summary(entry: dict, base: Path):
    path = base / entry["summary_json"]
    if not path.is_file():
        return None
    d = json.loads(path.read_text())
    if not _is_xeb_summary(d):
        return None
    return d


def load_xeb_per_circuit(entry: dict, base: Path):
    rel = entry.get("per_circuit_json")
    if not rel:
        return None
    path = base / rel
    if not path.is_file():
        return None
    return json.loads(path.read_text())


def _circuit_idx_from_stem(stem: str) -> int:
    m = re.search(r"_c(\d+)$", stem)
    return int(m.group(1)) if m else -1


def xeb_depth_stats(summary: dict) -> pd.DataFrame:
    pdms = summary.get("per_depth_mean_sem") or {}
    rows = []
    for dk in sorted(pdms.keys(), key=lambda x: int(x)):
        block = pdms[dk]
        rows.append(
            {
                "depth": int(dk),
                "mean_F": float(block["mean"]),
                "sem_F": float(block["sem"]),
                "n_circuits": int((summary.get("circuits_per_depth") or {}).get(str(dk), summary.get("num_circuits_per_depth", np.nan))),
            }
        )
    return pd.DataFrame(rows)


def xeb_per_circuit_long(entry: dict, base: Path) -> pd.DataFrame:
    per = load_xeb_per_circuit(entry, base)
    if not per:
        return pd.DataFrame()
    rows = []
    for depth_key, block in per.items():
        depth = int(depth_key)
        files = block.get("files") or []
        fids = block.get("fidelities") or []
        for fname, fid in zip(files, fids):
            stem = Path(fname).stem
            rows.append(
                {
                    "device_key": entry["device_key"],
                    "display_name": entry["display_name"],
                    "family": entry.get("family", "other"),
                    "depth": depth,
                    "circuit_stem": stem,
                    "circuit_idx": _circuit_idx_from_stem(stem),
                    "F_xeb": float(fid),
                }
            )
    return pd.DataFrame(rows)


xeb_summary_rows = []
xeb_per_circuit_frames = []
for entry in XEB_DEVICE_REGISTRY:
    summ = load_xeb_summary(entry, REPO_ROOT)
    if summ is None:
        print("SKIP XEB (missing or not xeb):", entry["summary_json"])
        continue
    stats = xeb_depth_stats(summ)
    d1 = stats.loc[stats["depth"] == 1]
    xeb_summary_rows.append(
        {
            "device_key": entry["device_key"],
            "display_name": entry["display_name"],
            "family": entry.get("family"),
            "backend": summ.get("backend"),
            "shots": summ.get("shots"),
            "depths": stats["depth"].tolist(),
            "F_d1_mean": float(d1["mean_F"].iloc[0]) if len(d1) else np.nan,
            "F_d1_sem": float(d1["sem_F"].iloc[0]) if len(d1) else np.nan,
            "n_circuits_d1": int(d1["n_circuits"].iloc[0]) if len(d1) and pd.notna(d1["n_circuits"].iloc[0]) else np.nan,
        }
    )
    pc = xeb_per_circuit_long(entry, REPO_ROOT)
    if len(pc):
        xeb_per_circuit_frames.append(pc)

xeb_agg = pd.DataFrame(xeb_summary_rows)
if xeb_agg.empty:
    raise RuntimeError("No XEB summaries loaded — check XEB_DEVICE_REGISTRY paths.")
order_x = [e["device_key"] for e in XEB_DEVICE_REGISTRY if e["device_key"] in set(xeb_agg["device_key"])]
xeb_agg["display_name"] = pd.Categorical(
    xeb_agg["display_name"], categories=[e["display_name"] for e in XEB_DEVICE_REGISTRY], ordered=True
)
xeb_agg = xeb_agg.sort_values("display_name").reset_index(drop=True)
xeb_agg.round({"F_d1_mean": 4, "F_d1_sem": 4})


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(xeb_agg))
pal = sns.color_palette("husl", n_colors=len(xeb_agg))
ax.bar(x, xeb_agg["F_d1_mean"], yerr=xeb_agg["F_d1_sem"], capsize=4, color=pal, edgecolor="black", linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels(xeb_agg["display_name"], rotation=30, ha="right")
ax.set_ylabel(r"$F_{\mathrm{XEB}}$ at depth 1 (mean $\pm$ SEM)")
ax.set_ylim(0, 1.08)
ax.axhline(1.0, color="gray", lw=0.8, alpha=0.45)
ax.set_title("Linear XEB fidelity (depth-1 random circuits, shared QASM set)")
plt.tight_layout()
plt.show()


In [ ]:
if xeb_per_circuit_frames:
    xeb_circ = pd.concat(xeb_per_circuit_frames, ignore_index=True)
    d1 = xeb_circ[xeb_circ["depth"] == 1].copy()
    plt.figure(figsize=(10, 4.5))
    sns.boxplot(data=d1, x="display_name", y="F_xeb", hue="display_name", dodge=False, legend=False)
    sns.stripplot(data=d1, x="display_name", y="F_xeb", color="black", alpha=0.35, size=4, jitter=0.15)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(r"$F_{\mathrm{XEB}}$ per circuit (depth 1)")
    plt.xlabel("")
    plt.title("Per-circuit XEB spread at depth 1")
    plt.tight_layout()
    plt.show()
else:
    xeb_circ = pd.DataFrame()
    print("No per_depth_per_circuit.json files loaded.")


In [ ]:
# Multi-depth decay curves (devices with >1 depth in summary)
curves = []
for entry in XEB_DEVICE_REGISTRY:
    summ = load_xeb_summary(entry, REPO_ROOT)
    if summ is None:
        continue
    stats = xeb_depth_stats(summ)
    if len(stats) < 2:
        continue
    curves.append(
        (
            entry["display_name"],
            stats["depth"].to_numpy(),
            stats["mean_F"].to_numpy(),
            stats["sem_F"].to_numpy(),
        )
    )

if curves:
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for name, depths, mean_f, sem_f in curves:
        ax.errorbar(depths, mean_f, yerr=sem_f, fmt="o-", capsize=4, ms=5, lw=1.6, label=name)
    ax.set_xlabel("circuit depth (XEB layer tag)")
    ax.set_ylabel(r"$F_{\mathrm{XEB}}$ (mean $\pm$ SEM)")
    ax.set_title("XEB fidelity vs depth (where multi-depth data exists)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No multi-depth XEB summaries in registry (only depth-1 bars above).")


In [ ]:
out_xeb = DATA_DIR / "xeb_multi_device_summary.csv"
xeb_agg.to_csv(out_xeb, index=False)
print("Wrote", out_xeb.resolve())

if len(xeb_circ):
    out_xeb_long = DATA_DIR / "xeb_multi_device_per_circuit_long.csv"
    xeb_circ.to_csv(out_xeb_long, index=False)
    print("Wrote", out_xeb_long.resolve(), "rows:", len(xeb_circ))
